# Train st0 (Relation Detection) — RoBERTa-large

First transformer in the multistep CausalSense pipeline.

**Task:** binary classification — does this sentence contain a causal relation? (cause / enable / intend / prevent vs. no_relation).

**Output:** a HuggingFace-format checkpoint folder you can later continue training on your own data via `AutoModelForSequenceClassification.from_pretrained(...)`.

**Runtime:** ~2–3 hr on a T4. Set Runtime → Change runtime type → T4 GPU before running.

## 0. Confirm GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU detected — switch runtime to GPU.'
print('Device:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 1. Install dependencies

In [ ]:
!pip install -q transformers==4.44.2 scikit-learn pandas tqdm

## 2. Clone the repo

Public repo — no auth needed.

In [ ]:
import os

if not os.path.isdir('Relation_extraction'):
    !git clone https://github.com/eitang5/Relation_extraction.git
else:
    print('Repo already cloned.')

%cd Relation_extraction

## 3. Mount Google Drive (recommended)

Checkpoints will survive Colab disconnects. If you skip this, change `OUTPUT_DIR` below to a local path like `checkpoints/st0_roberta_large` — but if the runtime dies mid-training, you lose it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/causalsense/checkpoints

## 4. Sanity run — 1 epoch on roberta-base (~3–5 min)

Confirms the pipeline works end-to-end before committing the full GPU time. Watch for:

- The script prints train/dev label distribution — should match `{1: ~92%, 0: ~8%}`.
- Validation classification report — both classes should have **nonzero precision and recall**. If class 0 is all zeros, the model is just predicting `1` for everything (expected at first, but a red flag if it stays that way after the real run).

Skip this cell if you're confident.

In [ ]:
import os
os.environ['DATA_DIR']    = 'separate_tasks/data'
os.environ['OUTPUT_DIR']  = 'checkpoints/st0_sanity'
os.environ['MODEL_NAME']  = 'roberta-base'
os.environ['EPOCHS']      = '1'
os.environ['BATCH_SIZE']  = '16'

!python separate_tasks/Relation_detection.py

## 5. Real run — roberta-large, 10 epochs (~2–3 hr on T4)

Output goes to Drive so the checkpoint survives Colab disconnects. The script saves the best dev-F1 model only.

**If you hit `CUDA out of memory`:** drop `BATCH_SIZE` to 4 (or 2) and re-run.

In [ ]:
import os
os.environ['DATA_DIR']    = 'separate_tasks/data'
os.environ['OUTPUT_DIR']  = '/content/drive/MyDrive/causalsense/checkpoints/st0_roberta_large'
os.environ['MODEL_NAME']  = 'roberta-large'
os.environ['EPOCHS']      = '10'
os.environ['BATCH_SIZE']  = '8'

!python separate_tasks/Relation_detection.py

## 6. Verify the checkpoint reloads

This is the format you'll fine-tune from when you have your own data.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

ckpt = os.environ['OUTPUT_DIR']
model = AutoModelForSequenceClassification.from_pretrained(ckpt)
tokenizer = AutoTokenizer.from_pretrained(ckpt)

print('Loaded:', ckpt)
print('Num labels:', model.config.num_labels)
print('Params (M):', round(sum(p.numel() for p in model.parameters()) / 1e6, 1))

## 7. Smoke-test inference

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.eval().to(device)

samples = [
    'The earthquake caused widespread destruction in coastal areas.',
    'She drank a cup of coffee.',
    'Heavy rain prevented the football match from taking place.',
    'The cat sat on the mat.',
]

for s in samples:
    inputs = tokenizer(s, return_tensors='pt', truncation=True, max_length=128).to(device)
    with torch.no_grad():
        pred = int(model(**inputs).logits.argmax(-1).item())
    label = 'has relation' if pred == 1 else 'no relation'
    print(f'[{label:>12}] {s}')

## Next steps

- This is **st0** — the first of 2–3 transformers in the multistep pipeline. Repeat with `separate_tasks/relation_classification.py` (st1, relation type) and optionally `separate_tasks/EE.py` (st2, event spans).
- **To continue training on your own data later:** set `MODEL_NAME` to the saved checkpoint folder path instead of `roberta-large`, point `DATA_DIR` at a folder with your `train.csv` / `dev.csv` / `test.csv` (same columns: `text`, `label` with values 0/1), and run the same cell.

Example:
```python
os.environ['MODEL_NAME'] = '/content/drive/MyDrive/causalsense/checkpoints/st0_roberta_large'
os.environ['DATA_DIR']   = '/content/drive/MyDrive/my_data'
os.environ['OUTPUT_DIR'] = '/content/drive/MyDrive/causalsense/checkpoints/st0_roberta_large_mydata'
os.environ['EPOCHS']     = '3'
!python separate_tasks/Relation_detection.py
```